# Лабораторна робота №4. Порівняння дерева рішень та ансамблів (Hill Valley)

## Етап 1. Дослідницьке питання і гіпотези

### Головне дослідницьке питання
Чи виправданий перехід від контрольованого дерева рішень до ансамблю (Random Forest / Gradient Boosting)
для задачі бінарної класифікації на наборі Hill Valley — з точки зору трьох властивостей одночасно:
**якість прогнозу**, **стійкість до зміни folds**, **можливість інтерпретації**?

### Гіпотези та критерії перевірки

| # | Гіпотеза | Показник | Правило висновку |
|---|----------|----------|------------------|
| H1 | Ансамбль (RF/GB) дає вищий середній **macro F1**, ніж контрольоване дерево. | Середнє macro F1 за 5-fold CV | H1 підтверджено, якщо різниця макрос-середніх ≥ 0.02 і 95% ДІ не перетинаються. |
| H2 | Random Forest менш чутливий до зміни folds, ніж одне дерево. | Std macro F1 між folds | H2 підтверджено, якщо std(RF) ≤ 0.5 × std(tree). |
| H3 | Обмеження глибини дерева зменшує розрив між train і validation якістю. | Δ = macroF1_train − macroF1_valid | H3 підтверджено, якщо Δ зменшується щонайменше на 50% при max_depth ≤ 5. |
| H4 | Ранги важливості ознак змінюються між folds, навіть якщо середня якість лишається високою. | Spearman ρ між важливостями ознак різних folds | H4 підтверджено, якщо медіанний ρ < 0.7. |

### Протокол (фіксується ДО першого навчання)
- **Джерело даних**: UCI ML Repository, Hill Valley dataset (файл `Hill_Valley_without_noise_Training.data`).
- **Основна метрика**: **macro F1** (бінарна задача, можливий дисбаланс; macro F1 враховує обидва класи однаково).
- **Розмір test set**: 20% зі стратифікацією, `random_state=42`.
- **CV**: StratifiedKFold, n_splits=5, shuffle=True, random_state=42.
- **Кандидати**:
  - DecisionTreeClassifier (max_depth ∈ {3, 5, 10, None}, min_samples_leaf ∈ {1, 5, 20})
  - RandomForestClassifier (n_estimators ∈ {100, 300, 500}, max_depth ∈ {None, 10}, max_features ∈ {sqrt, log2})
  - GradientBoostingClassifier (n_estimators ∈ {100, 300}, learning_rate ∈ {0.05, 0.1}, max_depth ∈ {2, 3})
- **Критерій вибору фінальної моделі**: найвищий середній macro F1 у внутрішньому CV із перевагою простоти при різниці < 0.005.
- **Test set** використовується **один раз** після завершення порівняння і вибору конфігурації.

In [1]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split

# --- Фіксована конфігурація експерименту ---
RANDOM_STATE = 42
TEST_SIZE = 0.20
TARGET_COL = "class"  # за потреби уточнимо після читання файлу

DATA_PATH = Path("Hill_Valley_without_noise_Training.data")
SPLIT_DIR = Path("data/splits")
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "cv_folds": 5,
    "primary_metric": "macro_f1",
    "data_path": str(DATA_PATH),
}

with open(SPLIT_DIR / "experiment_config.json", "w") as f:
    json.dump(CONFIG, f, indent=2, ensure_ascii=False)

CONFIG

{'random_state': 42,
 'test_size': 0.2,
 'cv_folds': 5,
 'primary_metric': 'macro_f1',
 'data_path': 'Hill_Valley_without_noise_Training.data'}

In [2]:
df = pd.read_csv(DATA_PATH)

print(f"Розмір: {df.shape}")
print(f"Колонки (перші 5): {list(df.columns[:5])}")
print(f"Колонки (останні 3): {list(df.columns[-3:])}")

# Визначаємо цільову колонку автоматично, якщо потрібно
if TARGET_COL not in df.columns:
    TARGET_COL = df.columns[-1]
    print(f"Цільова колонка визначена як: {TARGET_COL}")

feature_cols = [c for c in df.columns if c != TARGET_COL]
print(f"Кількість ознак: {len(feature_cols)}")
print(f"Цільова змінна: {TARGET_COL}, типи класів: {sorted(df[TARGET_COL].unique())}")

df.head(3)

Розмір: (606, 101)
Колонки (перші 5): ['X1', 'X2', 'X3', 'X4', 'X5']
Колонки (останні 3): ['X99', 'X100', 'class']
Кількість ознак: 100
Цільова змінна: class, типи класів: [np.int64(0), np.int64(1)]


,X1,X2,X3,X4,X5,X6,X7,X8,X9,X10,...,X92,X93,X94,X95,X96,X97,X98,X99,X100,class
0,1317.265789,1315.220951,1312.770581,1309.834252,1306.315588,1302.099102,1297.046401,1290.991646,1283.736109,1275.041652,...,1327.575109,1327.575350,1327.575552,1327.575719,1327.575859,1327.575976,1327.576074,1327.576155,1327.576223,0
1,7329.967624,7379.907443,7441.799231,7518.503422,7613.565031,7731.377492,7877.385707,8058.337694,8282.596458,8560.526497,...,7121.300474,7121.300438,7121.300410,7121.300387,7121.300368,7121.300353,7121.300341,7121.300331,7121.300323,1
2,809.421410,809.780119,810.207191,810.715653,811.321016,812.041748,812.899834,813.921452,815.137768,816.585886,...,807.545134,807.544181,807.543381,807.542709,807.542144,807.541670,807.541272,807.540937,807.540656,1


In [3]:
# 1) Розподіл класів
counts = df[TARGET_COL].value_counts().sort_index()
shares = df[TARGET_COL].value_counts(normalize=True).sort_index().round(4)
class_table = pd.DataFrame({"count": counts, "share": shares})
print("Розподіл класів:")
display(class_table)

# 2) Пропуски
missing = df.isna().sum()
missing_total = int(missing.sum())
print(f"Всього пропусків: {missing_total}")
if missing_total:
    display(missing[missing > 0])

# 3) Дублікати
dup_count = int(df.duplicated().sum())
print(f"Повних дублікатів рядків: {dup_count}")

# 4) Константні ознаки
const_feats = [c for c in feature_cols if df[c].nunique(dropna=False) <= 1]
print(f"Константні ознаки: {len(const_feats)} -> {const_feats[:10]}")

Розподіл класів:


,count,share
class,,
0,305,0.5033
1,301,0.4967


Всього пропусків: 0
Повних дублікатів рядків: 0
Константні ознаки: 0 -> []


In [4]:
# 1) Ознаки, що поводяться як ідентифікатори (унікальні для кожного об'єкта)
id_like = [c for c in feature_cols if df[c].nunique() == len(df)]
print(f"Ознаки, схожі на ID: {id_like if id_like else 'не виявлено'}")

# 2) Кореляція ознак із target (абсолютна)
target_numeric = df[TARGET_COL]
corr_with_target = df[feature_cols].corrwith(target_numeric).abs().sort_values(ascending=False)
print("Топ-10 ознак за |corr| з target:")
display(corr_with_target.head(10).round(3))

# 3) Ознаки з кореляцією > 0.95 — підозра на прямий витік
suspicious = corr_with_target[corr_with_target > 0.95]
print(f"Підозрілі ознаки (|corr| > 0.95): {list(suspicious.index)}")

# 4) Ознаки з нульовою дисперсією (ще раз перевіримо)
zero_var = [c for c in feature_cols if df[c].std() == 0]
print(f"Ознаки з нульовою дисперсією: {zero_var if zero_var else 'немає'}")

Ознаки, схожі на ID: ['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8', 'X9', 'X10', 'X11', 'X12', 'X13', 'X14', 'X15', 'X16', 'X17', 'X18', 'X19', 'X20', 'X21', 'X22', 'X23', 'X24', 'X25', 'X26', 'X27', 'X28', 'X29', 'X30', 'X31', 'X32', 'X33', 'X34', 'X35', 'X36', 'X37', 'X38', 'X39', 'X40', 'X41', 'X42', 'X43', 'X44', 'X45', 'X46', 'X47', 'X48', 'X49', 'X50', 'X51', 'X52', 'X53', 'X54', 'X55', 'X56', 'X57', 'X58', 'X59', 'X60', 'X61', 'X62', 'X63', 'X64', 'X65', 'X66', 'X67', 'X68', 'X69', 'X70', 'X71', 'X72', 'X73', 'X74', 'X75', 'X76', 'X77', 'X78', 'X79', 'X80', 'X81', 'X82', 'X83', 'X84', 'X85', 'X86', 'X87', 'X88', 'X89', 'X90', 'X91', 'X92', 'X93', 'X94', 'X95', 'X96', 'X97', 'X98', 'X99', 'X100']
Топ-10 ознак за |corr| з target:


X100    0.041
X99     0.041
X98     0.041
X1      0.040
X97     0.040
X2      0.040
X96     0.040
X3      0.040
X95     0.039
X4      0.039
dtype: float64

Підозрілі ознаки (|corr| > 0.95): []
Ознаки з нульовою дисперсією: немає


In [5]:
idx_train, idx_test = train_test_split(
    df.index.values,
    test_size=TEST_SIZE,
    stratify=df[TARGET_COL].values,
    random_state=RANDOM_STATE,
)

# Зберігаємо індекси — вони використовуються у всіх наступних етапах
pd.Series(idx_train, name="idx").to_csv(SPLIT_DIR / "train_idx.csv", index=False)
pd.Series(idx_test,  name="idx").to_csv(SPLIT_DIR / "test_idx.csv",  index=False)

print(f"Train: {len(idx_train)} об'єктів")
print(f"Test:  {len(idx_test)} об'єктів")
print(f"Індекси збережено у: {SPLIT_DIR}")

Train: 484 об'єктів
Test:  122 об'єктів
Індекси збережено у: data\splits


In [6]:
train_df = df.loc[idx_train]
test_df  = df.loc[idx_test]

split_check = pd.DataFrame({
    "train_count": train_df[TARGET_COL].value_counts().sort_index(),
    "train_share": train_df[TARGET_COL].value_counts(normalize=True).sort_index().round(4),
    "test_count":  test_df[TARGET_COL].value_counts().sort_index(),
    "test_share":  test_df[TARGET_COL].value_counts(normalize=True).sort_index().round(4),
})
print("Перевірка стратифікації:")
display(split_check)

# Додаткова перевірка: індекси не перетинаються
assert len(np.intersect1d(idx_train, idx_test)) == 0, "Train і Test перетинаються!"
assert len(idx_train) + len(idx_test) == len(df), "Втрачені або зайві індекси!"
print("OK: поділ коректний, train і test не перетинаються.")

Перевірка стратифікації:


,train_count,train_share,test_count,test_share
class,,,,
0,244,0.5041,61,0.5
1,240,0.4959,61,0.5


OK: поділ коректний, train і test не перетинаються.


## Підсумок Етапу 1–2

**Дослідницьке питання** і 4 гіпотези (H1–H4) зафіксовані у Клітинці 1.

**Що зроблено:**
- завантажено Hill Valley (без шуму);
- перевірено розмір, розподіл класів, пропуски, дублікати, константи;
- виконано перевірку на data leakage (ID-подібні ознаки, кореляції з target);
- виконано **стратифікований** поділ train/test (80/20, `random_state=42`);
- збережено `train_idx.csv` / `test_idx.csv` та `experiment_config.json`.

**Наступні етапи** (Етап 3+): побудова `Pipeline`, nested CV для налаштування гіперпараметрів,
порівняння дерева vs RF vs GB за macro F1, аналіз стійкості (std між folds) та стабільності
важливості ознак (Spearman ρ). Test set використовується **один раз** у фіналі.